# MicroDuck — 이전 학습 checkpoint 정리

기존 **`jump_spin`** 실험에서 최종 `.pt` 하나를 보존하고 중간 `model_*.pt`만 정리합니다.
새 **`jump_spin_launch_v2`**, 영상, 평가 JSON, 설정 파일, 학습 로그는 삭제하지 않습니다.
GPU와 저장소 설치가 필요하지 않습니다. Colab의 기본 CPU 런타임으로 실행하세요.

1. 아래 셀에서 Google Drive를 연결합니다.
2. 보존할 checkpoint 경로를 확인합니다. 기본값은 이전 실험의 `model_9999.pt`입니다.
3. `DELETE = False`로 실행해 삭제 대상과 예상 확보 용량을 확인합니다.
4. 실제 삭제하려면 `DELETE = True`로 바꿔 **설정 셀과 정리 셀을 다시 실행**합니다.

최종 파일이 없거나 비어 있으면 삭제하지 않고 중단합니다. 파일 존재·크기 확인은
모델 내용의 무결성 검사와는 다릅니다. 실행 중인 학습이 없는 상태에서 정리하세요.
삭제 셀은 파일을 직접 삭제하므로 Drive 휴지통으로 이동한다고 가정하지 마세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 보존할 파일 설정

`KEEP_RELATIVE_PATH`는 `logs/rsl_rl/jump_spin/` 아래의 상대 경로입니다.
다른 실행의 최종 모델을 보존하려면 실행 폴더명과 파일명을 수정하세요.
기본 모델은 기존 실패 실험의 최종 모델이며, 새 학습 결과와 별개로 보관됩니다.


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/microduck-training"
KEEP_RELATIVE_PATH = "2026-10-08_12-32-56_colab-a100/model_9999.pt"
DELETE = False  # 미리보기 확인 후 실제 삭제할 때만 True로 변경


In [ ]:
from pathlib import Path

# 정리 범위는 이전 실험 jump_spin으로 제한합니다.
root = Path(DRIVE_ROOT) / "logs" / "rsl_rl" / "jump_spin"
if root.is_symlink():
    raise ValueError(f"실험 폴더가 심볼릭 링크입니다. 실제 경로를 확인하세요: {root}")
root = root.resolve()
keep = (root / KEEP_RELATIVE_PATH).resolve()
keep.relative_to(root)  # 실험 폴더 바깥의 보존 경로는 거부합니다.

if not keep.is_file() or keep.stat().st_size == 0:
    raise RuntimeError(f"보존할 최종 파일을 확인할 수 없습니다: {keep}")
if not keep.match("model_*.pt"):
    raise ValueError(f"보존 파일은 model_*.pt checkpoint여야 합니다: {keep}")

targets = sorted(
    p for p in root.rglob("model_*.pt")
    if p.is_file() and p.resolve() != keep
)
size_gb = sum(p.stat().st_size for p in targets) / 1024**3

print("보존:", keep)
print(f"삭제 대상: {len(targets)}개 / 약 {size_gb:.2f} GB")
for p in targets:
    print(p)

if DELETE:
    for p in targets:
        p.unlink()
    print(f"중간 checkpoint {len(targets)}개 삭제 완료")
    print("최종 checkpoint 보존:", keep)
else:
    print("아직 삭제하지 않았습니다.")
    print("확인 후 DELETE = True로 바꾸고 설정 셀과 정리 셀을 다시 실행하세요.")
